# GRAM ablations: marginal averages

参考所提供的两张图：差值采用两个 noise-rate 面板的蓝–白–橙热图；趋势采用六数据集小多图、serif 字体、粗边框、marker 和均值 ± SD 阴影。

**聚合口径**：始终保留 dataset、noise rate 和当前控制变量。每个有效配置先平均实际可用 seeds，再等权平均未展示的参数，最后等权平均 noise type。配对差值先在相同训练、噪声、预算及共有 seed 上计算，再聚合。阴影为完成参数平均后的三个 noise type 之间的样本 SD，不是置信区间。

- β 敏感性：完整 adaptive 核、repair + IV，平均全部五个 k。
- k 敏感性：完整 adaptive 核、repair + IV，平均全部七个 β。
- 组件 / 权重与 acquisition 消融：平均全部七个 β；现有网格只在各数据集参考 k 上运行这些消融，因此不声称已平均未运行的 k。β 无关的策略按逻辑 β 与相应完整模型配对，不把重复引用当成独立运行。
- kernel weight 轨迹：完整 adaptive 核、repair + IV，保留 verification budget，平均 β、k、noise type、seed。

沿用完整网格的指标口径：从首个预算点到该 noise rate 的最大预算，按实际 verification fraction 梯形积分并除以区间长度。两个 noise rate 分别展示，不相互平均。
所有来源、配对 seed 和聚合数据保留在 notebook 内存中；磁盘仅导出 PDF 至 `analysis/figures/ablation/grid/`。

In [ ]:
from pathlib import Path
from itertools import product
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import PercentFormatter, ScalarFormatter
from matplotlib.lines import Line2D
from IPython.display import Markdown, display
from IPython import get_ipython

if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')

DATASETS = ['cifar10', 'cifar100', 'atis', 'qnli', 'adult', 'letter']
NOISE_TYPES = ['symmetric', 'pairflip', 'instance']
RHOS = [0.2, 0.4]
SEEDS = [0, 1, 2]
FOCUS_DATASET, FOCUS_NOISE, FOCUS_RHO = 'qnli', 'instance', 0.2
COMMON_START, COMMON_END = 0.001, 0.20
REPORT_BUDGETS = [0.01, 0.05, 0.10, 0.20]
TABLES = {}  # Full table data stays in notebook memory.

def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'config/methods.py').is_file() and (candidate / 'src/gram/policy.py').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or analysis/.')

ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from config.methods import DEFAULT_GRAM_VARIANT, GRAM_VARIANTS, method_config
from src.utils.run_logging import completed_run_exists

MAIN = DEFAULT_GRAM_VARIANT
FAMILIES = {
    'Components / weights': [MAIN, 'abl_fixed_uniform_weights', 'abl_no_identity',
                             'abl_no_gradient', 'abl_no_margin'],
    'Graph neighbors': ['abl_k10', 'abl_k25', MAIN, 'abl_k100', 'abl_k250'],
    'Acquisition': [MAIN, 'abl_risk_only', 'abl_iv_only',
                    'abl_variance_only', 'abl_local_only'],
    'IV beta': ['abl_beta0p1', MAIN, 'abl_beta10', 'abl_beta100', 'abl_beta1000', 'abl_beta10000'],
}
VARIANTS = [MAIN] + [v for group in FAMILIES.values() for v in group if v != MAIN]
LABELS = {
    MAIN: 'GRAM (Ours), p + IV', 'abl_fixed_uniform_weights': 'Fixed uniform weights',
    'abl_no_identity': 'No identity', 'abl_no_gradient': 'No gradient',
    'abl_no_margin': 'No margin',
    **{f'abl_k{k}': f'k = {k}' for k in [10, 25, 100, 250]},
    'abl_risk_only': 'Risk only (p)', 'abl_iv_only': 'IV only',
    'abl_variance_only': 'Variance only', 'abl_local_only': 'p + local IV',
    **{f'abl_beta{str(b).replace(".", "p")}': f'IV beta = {b}' for b in [0.1, 10, 100, 1000, 10000]},
}
FAMILY_OF = {v: family for family, variants in FAMILIES.items() for v in variants if v != MAIN}
FAMILY_OF[MAIN] = 'Reference'
SCENARIO = ['dataset', 'noise_type', 'rho']
KEY = SCENARIO + ['seed']
RESULTS = ROOT / 'results/detection'
SNAPSHOT_UTC = datetime.now(timezone.utc).isoformat(timespec='seconds')
assert 0 < COMMON_START < COMMON_END <= min(RHOS)
assert FOCUS_DATASET in DATASETS and FOCUS_NOISE in NOISE_TYPES and FOCUS_RHO in RHOS
assert set(VARIANTS) <= set(GRAM_VARIANTS)
pd.set_option('display.max_colwidth', 65)
pd.set_option('display.max_rows', 60)

def record_table(frame, name):
    TABLES[name] = frame.copy()

def show_table(frame, name, caption):
    record_table(frame, name)
    display(frame.style.hide(axis='index').set_caption(caption).set_properties(**{
        'font-size': '12px', 'white-space': 'nowrap', 'padding': '5px 8px',
    }))

def fmt(values, scale=100, digits=2):
    vals = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if vals.empty:
        return '—'
    mean = vals.mean() * scale
    if len(vals) == 1:
        return f'{mean:.{digits}f} (n=1)'
    return f'{mean:.{digits}f} ± {vals.std(ddof=1) * scale:.{digits}f} (n={len(vals)})'

def focus(frame):
    if frame.empty:
        return frame.copy()
    return frame[(frame.dataset == FOCUS_DATASET) & (frame.noise_type == FOCUS_NOISE)
                 & np.isclose(frame.rho, FOCUS_RHO)].copy()

def close_to(series, value):
    return np.isclose(pd.to_numeric(series, errors='coerce').to_numpy(dtype=float), value)

display(Markdown(f'Data snapshot: **{SNAPSHOT_UTC}**.'))

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.labelweight': 'bold',
    'axes.linewidth': 1.6,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.color': '#BFBFBF',
    'grid.linestyle': ':',
    'grid.linewidth': 0.65,
    'grid.alpha': 0.55,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.major.width': 1.5,
    'ytick.major.width': 1.5,
    'legend.fontsize': 9,
    'figure.titlesize': 15,
    'figure.titleweight': 'bold',
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'svg.fonttype': 'none',
})
DATASET_LABELS = {
    'cifar10': 'CIFAR-10', 'cifar100': 'CIFAR-100', 'atis': 'ATIS',
    'qnli': 'QNLI', 'adult': 'Adult', 'letter': 'Letter',
}
COMPONENT_COLORS = {'margin': '#0072B2', 'gradient': '#009E73', 'identity': '#CC79A7'}
COMPONENT_MARKERS = {'margin': 'o', 'gradient': 's', 'identity': '^'}
DATASET_COLORS = {
    'cifar10': '#0072B2', 'cifar100': '#D55E00', 'atis': '#009E73',
    'qnli': '#CC79A7', 'adult': '#E69F00', 'letter': '#222222',
}
DATASET_MARKERS = {
    'cifar10': 'o', 'cifar100': 's', 'atis': '^',
    'qnli': 'D', 'adult': 'P', 'letter': 'X',
}
FIGURE_MANIFEST = []
FIGURE_DATA = {}

def finish_figure(fig, stem, data, caption, output_dir=None):
    output_dir = ROOT / 'analysis/figures/ablation' if output_dir is None else output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f'{stem}.pdf'
    fig.savefig(path, format='pdf', bbox_inches='tight', pad_inches=0.10, facecolor='white')
    FIGURE_DATA[stem] = data.copy()
    FIGURE_MANIFEST.append({'figure': stem, 'data_rows': len(data), 'caption': caption})
    plt.show()
    plt.close(fig)
    return path

plt.rcParams['pdf.fonttype'] = 42

## Load and validate the complete grid
Read completed runs only; check configurations, budgets, and shared training fingerprints.

In [ ]:
from config.methods import (GRAM_DATASET_NEIGHBORS, GRAM_GRID_NEIGHBORS,
                            gram_grid_variants, method_config as effective_method_config)
PROJECT_ROOT, RESULTS_ROOT = ROOT, RESULTS
GRID_OUTPUT_DIR = ROOT / 'analysis/figures/ablation/grid'
GRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_METHODS = []
REQUIRED_SEEDS = [0]
GRAM_BETAS = [0.0, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
GRAM_COLOR = '#D55E00'
GRID_VARIANTS_BY_DATASET = {dataset: gram_grid_variants(reference_neighbors=GRAM_DATASET_NEIGHBORS[dataset])
                            for dataset in DATASETS}
EXPECTED_BUDGETS = {rho: np.array([0.001,0.005,0.01,0.025] +
                    [round(.05*i,12) for i in range(1,round(rho/.05)+1)]) for rho in RHOS}

In [ ]:
# Read every complete seed for each expected per-dataset configuration.
# Exact directory depths exclude source_metadata and shared training artifacts.
REQUIRED_FILES = ("run_args.json", "config.json", "metrics.csv", "queries.csv",
                  "predictions.csv", "summary.json")
REQUIRED_COLUMNS = ["configured_budget_fraction", "verification_fraction",
                    "unverified_auprc", "query_precision"]
RUN_KEYS = ["dataset", "noise_type", "rho", "setting", "seed"]
run_rows, all_checkpoint_frames = [], []
expected_configs = {
    variant: json.loads(json.dumps(effective_method_config("ours", variant)))
    for variants in GRID_VARIANTS_BY_DATASET.values() for variant in variants
}
expected_runs = []

run_specs, excluded_seed_dirs = [], []
for dataset in DATASETS:
    candidates = [("ours", variant) for variant in GRID_VARIANTS_BY_DATASET[dataset]]
    candidates += [(method, "") for method in BASELINE_METHODS]
    for (method, variant), noise_type, rho in product(candidates, NOISE_TYPES, RHOS):
        base = RESULTS_ROOT / dataset / noise_type / f"rho{rho:g}" / method
        if method == "ours":
            base /= variant
        available = []
        for directory in sorted(base.glob("seed*")):
            suffix = directory.name.removeprefix("seed")
            if not directory.is_dir() or not suffix.isdigit():
                continue
            seed = int(suffix)
            if completed_run_exists(directory, protocol="full_pool_detection", required_files=REQUIRED_FILES):
                available.append(seed)
                run_specs.append((dataset, method, variant, noise_type, rho, seed, directory))
            else:
                excluded_seed_dirs.append({"source": directory.relative_to(PROJECT_ROOT).as_posix(),
                                           "status": "incomplete seed excluded"})
        if not set(REQUIRED_SEEDS) <= set(available):
            raise ValueError(f"Required completed seed is missing: {base}; observed={available}")

for dataset, method, variant, noise_type, rho, seed, directory in run_specs:
    is_gram = method == "ours"
    if not completed_run_exists(directory, protocol="full_pool_detection", required_files=REQUIRED_FILES):
        raise ValueError(f"Missing/incomplete plotting run: {directory}")
    metrics_path = directory / "metrics.csv"
    config = json.loads((directory / "config.json").read_text())
    summary = json.loads((directory / "summary.json").read_text())
    identity = dict(dataset=dataset, noise_type=noise_type, rho=rho, method=method, seed=seed)
    if any(summary.get(key) != value for key, value in identity.items()):
        raise ValueError(f"Summary identity mismatch: {directory}")
    if (config.get("dataset") != dataset or config.get("method") != method
        or config.get("protocol") != "full_pool_detection"
        or summary.get("protocol") != "full_pool_detection"
        or any(config["scenario"].get(k) != identity[k] for k in ("noise_type", "rho", "seed"))):
        raise ValueError(f"Configuration identity mismatch: {directory}")
    mconfig = config["method_config"]
    if is_gram and (config.get("gram_variant") != variant
                    or summary.get("gram_variant") != variant
                    or mconfig != expected_configs[variant]):
        raise ValueError(f"GRAM effective configuration mismatch: {directory}")
    frame = pd.read_csv(metrics_path).sort_values("configured_budget_fraction")
    expected = EXPECTED_BUDGETS[rho]
    configured = frame["configured_budget_fraction"].to_numpy(float)
    if (len(configured) != len(expected) or not np.allclose(configured, expected)
        or not np.allclose(configured, summary["budgets"])
        or not np.allclose(configured, config["scenario"]["budgets"])):
        raise ValueError(f"Missing/unexpected budget checkpoints: {metrics_path}")
    values = frame[REQUIRED_COLUMNS].to_numpy(float)
    if not np.isfinite(values).all() or not ((values >= 0) & (values <= 1)).all():
        raise ValueError(f"Nonfinite/out-of-range metrics: {metrics_path}")
    counts = frame["verification_budget"].to_numpy(int)
    if (not np.all(np.diff(counts) > 0)
        or counts[-1] != summary["actual_budget"]
        or counts.tolist() != config["budget_checkpoint_target_counts"]):
        raise ValueError(f"Actual query budget mismatch: {metrics_path}")
    x = counts / summary["sample_count"]
    if not np.allclose(x, frame["verification_fraction"], atol=1e-5):
        raise ValueError(f"Verification fractions do not match counts: {metrics_path}")
    precision = frame["queried_mislabels_found"].to_numpy(int) / counts
    if not np.allclose(precision, frame["query_precision"], atol=1e-5):
        raise ValueError(f"Query precision does not match counts: {metrics_path}")
    frame["verification_fraction"] = x
    frame["query_precision"] = precision
    y = frame[["unverified_auprc", "query_precision"]].to_numpy(float)
    budget_means = np.trapz(y, x=x, axis=0) / (x[-1] - x[0])
    fingerprint = summary.get("shared_training_artifact_fingerprint")
    if is_gram and (not fingerprint or fingerprint != config["training_protocol"].get("shared_training_artifact_fingerprint")):
        raise ValueError(f"Missing/mismatched training fingerprint: {directory}")
    context = json.dumps({
        "dataset_config": config["dataset_config"], "model_config": config["model_config"],
        "sample_count": summary["sample_count"], "errors": summary["injected_mislabel_count"],
        "budgets": configured.tolist(), "query_counts": counts.tolist(),
        "fingerprint": fingerprint,
    }, sort_keys=True)
    setting = f"ours/{variant}" if is_gram else method
    record = {
        **identity, "setting": setting, "gram_variant": variant,
        "auprc": budget_means[0], "precision": budget_means[1],
        "budget_start": x[0], "budget_end": x[-1],
        "max_budget_configured": rho, "checkpoint_count": len(frame),
        "reference_k": GRAM_DATASET_NEIGHBORS[dataset] if is_gram else np.nan,
        "graph_neighbors": mconfig.get("graph_neighbors", np.nan),
        "acquisition": mconfig.get("acquisition", ""),
        "iv_beta": mconfig.get("iv_beta", np.nan),
        "kernel_components": "+".join(mconfig.get("active_kernel_components", [])),
        "kernel_weights": mconfig.get("kernel_weights", ""),
        "training_fingerprint": fingerprint, "pairing_context": context,
        "historical_result_reused": (directory / "result_reuse.json").is_file(),
        "training_fingerprint_overridden": "fingerprint_override" in summary,
        "source": metrics_path.relative_to(PROJECT_ROOT).as_posix(),
    }
    run_rows.append(record)
    expected_runs.append(tuple(record[key] for key in RUN_KEYS))
    checkpoints = frame[["checkpoint", "configured_budget_fraction", "verification_fraction",
                         "unverified_auprc", "query_precision"]].rename(
        columns={"unverified_auprc": "auprc", "query_precision": "precision"})
    for name, value in record.items():
        if name not in ("auprc", "precision"):
            checkpoints[name] = value
    if is_gram:
        checkpoints["beta"] = mconfig["iv_beta"]
        all_checkpoint_frames.append(checkpoints)

run_summaries = pd.DataFrame(run_rows)
observed_runs = pd.MultiIndex.from_frame(run_summaries[RUN_KEYS])
expected_index = pd.MultiIndex.from_tuples(expected_runs, names=RUN_KEYS)
if observed_runs.has_duplicates or not expected_index.equals(observed_runs):
    raise ValueError("Duplicate or incomplete comparison grid")
SEEDS = sorted(run_summaries.seed.unique().tolist())
assert len(run_summaries) == len(run_specs)
grid_checkpoints = pd.concat(all_checkpoint_frames, ignore_index=True)
grid_runs = run_summaries[run_summaries.method.eq("ours")].copy()
pair_keys = ["dataset", "noise_type", "rho", "seed"]
if not grid_runs.groupby(pair_keys)["pairing_context"].nunique().eq(1).all():
    raise ValueError("GRAM controls are not paired on training/noise/budget context")
seed_coverage = (run_summaries.groupby(["dataset", "noise_type", "rho", "method", "gram_variant", "setting"],
                                     as_index=False, dropna=False)
                 .agg(seed_count=("seed", "nunique"), seeds=("seed", lambda x: ",".join(map(str, sorted(x.unique()))))))
grid_inventory = (grid_runs.groupby("dataset", as_index=False)
                  .agg(configurations=("gram_variant", "nunique"), completed_runs=("setting", "size"),
                       reference_k=("reference_k", "first")))
coverage_summary = (seed_coverage[seed_coverage.method.eq("ours")].groupby("dataset", as_index=False)
                    .agg(configuration_scenarios=("seed_count", "size"),
                         seed_count_min=("seed_count", "min"), seed_count_max=("seed_count", "max")))
grid_inventory = grid_inventory.merge(coverage_summary, on="dataset", validate="one_to_one")
assert grid_inventory.configurations.eq(71).all() and grid_inventory.configuration_scenarios.eq(426).all()
assert grid_inventory.seed_count_min.ge(1).all()
# Context JSON is checked in memory; source paths and fingerprints remain in the notebook DataFrames.
print(f"Loaded {len(grid_runs):,} GRAM grid runs and {len(run_summaries)-len(grid_runs):,} baseline runs; available seeds={SEEDS}.")
print("Each configuration averages all its completed seeds before configuration/checkpoint selection.")
display(grid_inventory)

## Marginal averages
Average available seeds within each configuration before averaging nuisance parameter settings and noise types. Each noise type has equal weight.

In [ ]:
# Build logical comparisons before averaging nuisance parameters.
FAMILY_SETTINGS = {
    "Beta": [("Full model", "adaptive", "repair_iv")],
    "Neighbors": [(str(k), "adaptive", "repair_iv") for k in GRAM_GRID_NEIGHBORS],
    "Components": [("Fixed uniform", "fixed_uniform_weights", "repair_iv"),
                   ("No identity", "no_identity", "repair_iv"),
                   ("No gradient", "no_gradient", "repair_iv"),
                   ("No margin", "no_margin", "repair_iv")],
    "Acquisition": [("Risk only", "adaptive", "risk_only"),
                    ("IV only", "adaptive", "iv_only"),
                    ("Variance only", "adaptive", "variance_only"),
                    ("Risk + local IV", "adaptive", "local_only")],
}
METRIC_LABELS = {"auprc": "Budget-averaged AUPRC", "precision": "Budget-averaged query precision"}
grid_lookup = grid_runs.set_index(["dataset", "noise_type", "rho", "gram_variant", "seed"]).sort_index()
family_records = []
join_sources = lambda frame: " | ".join(sorted(frame.source))
for dataset, noise, rho, beta in product(DATASETS, NOISE_TYPES, RHOS, GRAM_BETAS):
    reference_k = GRAM_DATASET_NEIGHBORS[dataset]
    for family, settings in FAMILY_SETTINGS.items():
        for label, kernel, acquisition in settings:
            ks = GRAM_GRID_NEIGHBORS if family == "Beta" else [int(label) if family == "Neighbors" else reference_k]
            for k in ks:
                full_variant, = gram_grid_variants(betas=[beta], neighbors=[k], kernels=["adaptive"],
                                                 acquisitions=["repair_iv"], reference_neighbors=reference_k)
                variant, = gram_grid_variants(betas=[beta], neighbors=[k], kernels=[kernel],
                                             acquisitions=[acquisition], reference_neighbors=reference_k)
                rows = grid_lookup.loc[(dataset, noise, rho, variant)].sort_index()
                reference = grid_lookup.loc[(dataset, noise, rho, full_variant)].sort_index()
                common = rows.index.intersection(reference.index).sort_values()
                if len(common) == 0:
                    raise ValueError(f"No paired seeds: {dataset}, {noise}, {rho}, {variant}")
                paired, paired_reference = rows.loc[common], reference.loc[common]
                if not np.array_equal(paired.pairing_context.to_numpy(), paired_reference.pairing_context.to_numpy()):
                    raise ValueError(f"Unpaired grid comparison: {variant}")
                family_records.append({
                    "dataset": dataset, "noise_type": noise, "rho": rho, "beta": beta,
                    "family": family, "control": label, "graph_neighbors": k, "reference_k": reference_k,
                    "gram_variant": variant, "reference_variant": full_variant,
                    "auprc": rows.auprc.mean(), "precision": rows.precision.mean(),
                    "delta_auprc": (paired.auprc-paired_reference.auprc).mean(),
                    "delta_precision": (paired.precision-paired_reference.precision).mean(),
                    "control_seed_count": len(rows), "paired_seed_count": len(common),
                    "control_seeds": ",".join(map(str, rows.index)), "paired_seeds": ",".join(map(str, common)),
                    "source": join_sources(rows), "reference_source": join_sources(reference),
                    "paired_sources": join_sources(paired), "paired_reference_sources": join_sources(paired_reference),
                })
grid_family_runs = pd.DataFrame(family_records)
represented_sources = {path for value in grid_family_runs.source for path in value.split(" | ")}
assert represented_sources == set(grid_runs.source)
assert not grid_family_runs.duplicated(["dataset", "noise_type", "rho", "beta", "family", "control", "graph_neighbors"]).any()

SCORES = ["auprc", "precision", "delta_auprc", "delta_precision"]

def marginal_summary(family, control):
    """Each available seed -> equal nuisance settings -> equal noise types."""
    rows = grid_family_runs[grid_family_runs.family.eq(family)]
    keys = ["dataset", "rho", control]
    per_noise = rows.groupby(keys + ["noise_type"], as_index=False, observed=True)[SCORES].mean()
    summary = per_noise.groupby(keys, observed=True)[SCORES].agg(["mean", "std"]).reset_index()
    summary.columns = ["_".join(x).rstrip("_") if isinstance(x, tuple) else x for x in summary.columns]
    counts = per_noise.groupby(keys, observed=True).size().rename("noise_count").reset_index()
    coverage = rows.groupby(keys, observed=True).agg(
        beta_count=("beta", "nunique"), k_count=("graph_neighbors", "nunique"),
        seed_count_min=("control_seed_count", "min"), seed_count_max=("control_seed_count", "max"),
        paired_seed_count_min=("paired_seed_count", "min"), paired_seed_count_max=("paired_seed_count", "max"),
    ).reset_index()
    summary = summary.merge(counts, on=keys, validate="one_to_one").merge(coverage, on=keys, validate="one_to_one")
    assert summary.noise_count.eq(len(NOISE_TYPES)).all()
    record_table(per_noise, f"{family.lower()}_marginal_per_noise")
    record_table(summary, f"{family.lower()}_marginal_summary")
    return summary

MARGINALS = {family: marginal_summary(family, "beta" if family == "Beta" else "control")
             for family in FAMILY_SETTINGS}
assert MARGINALS["Beta"].k_count.eq(len(GRAM_GRID_NEIGHBORS)).all()
for family in ["Neighbors", "Components", "Acquisition"]:
    assert MARGINALS[family].beta_count.eq(len(GRAM_BETAS)).all()
assert MARGINALS["Components"].k_count.eq(1).all()
assert MARGINALS["Acquisition"].k_count.eq(1).all()
display(pd.concat([frame.assign(family=family) for family, frame in MARGINALS.items()], ignore_index=True))
print(f"All {grid_runs.source.nunique():,} grid runs represented; paired comparisons and marginal coverage verified.")

In [ ]:
GRID_PDF_PATHS = []

def finish_marginal_figure(fig, stem, data, caption):
    path = finish_figure(fig, stem, data, caption, output_dir=GRID_OUTPUT_DIR)
    GRID_PDF_PATHS.append(path)
    print(path.relative_to(PROJECT_ROOT).as_posix())
    return path

def plot_marginal_heatmap(family, metric, controls=None, stem=None, title=None):
    values = MARGINALS[family]
    controls = controls or [setting[0] for setting in FAMILY_SETTINGS[family]]
    matrices = [values[np.isclose(values.rho, rho)].pivot(index="control", columns="dataset", values=f"delta_{metric}_mean")
                .reindex(index=controls, columns=DATASETS) * 100 for rho in RHOS]
    assert all(np.isfinite(matrix.to_numpy()).all() for matrix in matrices)
    limit = max(1., max(np.abs(matrix.to_numpy()).max() for matrix in matrices))
    cmap = LinearSegmentedColormap.from_list("blue_white_orange", ["#2166AC", "#F7F7F7", "#B35806"])
    norm = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 1.15 + .56 * len(controls)), layout="constrained")
    for ax, rho, matrix in zip(axes, RHOS, matrices):
        image = ax.imshow(matrix.to_numpy(), cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")
        ax.set_xticks(range(len(DATASETS)), [DATASET_LABELS[d] for d in DATASETS], rotation=30, ha="right")
        ax.set_yticks(range(len(controls)), controls)
        ax.set_title(f"rho={rho:g}")
        ax.grid(False)
        for r in range(len(controls)):
            for c in range(len(DATASETS)):
                value = matrix.iloc[r, c]
                red, green, blue, _ = cmap(norm(value))
                luminance = .2126*red + .7152*green + .0722*blue
                ax.text(c, r, f"{value:+.2f}\n(n={len(NOISE_TYPES)})", ha="center", va="center", fontsize=8,
                        color="white" if luminance < .48 else "#222222")
    bar = fig.colorbar(image, ax=axes, shrink=.82, pad=.025)
    bar.set_label("Ablation - full model (pp)", fontsize=11)
    fig.suptitle(f"{title or family} | {METRIC_LABELS[metric]}", fontsize=15)
    plotted = values[values.control.isin(controls)].copy()
    return finish_marginal_figure(fig, stem or f"marginal_{family.lower()}_delta_{metric}", plotted,
        "Paired percentage-point change; averaged over available shared seeds, beta and noise types. "
        "Only the available reference k is used. n counts noise types, not seeds or beta settings.")

def plot_marginal_sensitivity(family, rho):
    control = "beta" if family == "Beta" else "control"
    order = GRAM_BETAS if family == "Beta" else [str(k) for k in GRAM_GRID_NEIGHBORS]
    values = MARGINALS[family]
    values = values[np.isclose(values.rho, rho)]
    fig, axes = plt.subplots(2, 3, figsize=(8.6, 5.1), sharex=True, sharey=True, layout="constrained")
    for index, (ax, dataset) in enumerate(zip(axes.flat, DATASETS)):
        panel = values[values.dataset.eq(dataset)].set_index(control).reindex(order)
        x = np.arange(len(order))
        for metric, color, marker in [("auprc", "#0072B2", "o"), ("precision", "#009E73", "s")]:
            mean, std = panel[f"{metric}_mean"].to_numpy(float), panel[f"{metric}_std"].to_numpy(float)
            ax.plot(x, mean, color=color, marker=marker, markersize=4, linewidth=2, label=METRIC_LABELS[metric])
            ax.fill_between(x, np.clip(mean-std, 0, 1), np.clip(mean+std, 0, 1), color=color, alpha=.11)
        ax.set_title(DATASET_LABELS[dataset])
        ax.set_xticks(x, [f"{float(v):g}" for v in order], rotation=30 if family == "Beta" else 0)
        ax.set_ylim(0, 1)
        ax.set_xlabel(r"IV weight $\beta$" if family == "Beta" else "Graph neighbors k")
        if index % 3 == 0:
            ax.set_ylabel("Budget-averaged score")
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, ncol=2, loc="outside lower center", frameon=False)
    fig.suptitle(f"{'IV beta' if family == 'Beta' else 'Graph neighbors'} sensitivity | rho={rho:g}")
    return finish_marginal_figure(fig, f"marginal_{family.lower()}_rho{rho:g}", values,
        "Available seeds are averaged per configuration; nuisance parameters are averaged within each noise type. "
        "Lines show equal-noise-type means; bands show SD across noise types, clipped to [0, 1].")

## Component, weight and acquisition effects
Each heatmap retains only dataset, noise rate and the ablation control. Values are paired percentage-point changes from the matching full model; n=3 is the number of noise types.

In [ ]:
for metric in ["auprc", "precision"]:
    plot_marginal_heatmap("Components", metric,
        controls=["No identity", "No gradient", "No margin"],
        stem=f"marginal_component_removal_delta_{metric}", title="Component removal")
    plot_marginal_heatmap("Components", metric, controls=["Fixed uniform"],
        stem=f"marginal_fixed_uniform_delta_{metric}", title="Fixed uniform weights")
    plot_marginal_heatmap("Acquisition", metric)

## Parameter sensitivity
β curves average k; k curves average β. Both average seeds and noise types. Bands show SD across noise types after the other averages.

In [ ]:
for rho in RHOS:
    plot_marginal_sensitivity("Beta", rho)
    plot_marginal_sensitivity("Neighbors", rho)

## Adaptive kernel weights
Keep the verification-budget axis and average β, k, seeds and noise types. All complete adaptive repair + IV runs in the parameter grid contribute.

In [ ]:
# Complete adaptive repair+IV grid only: average beta and k as nuisance variables.
# Resolve beta=0 aliases through the registry, then count each logical configuration once.
weight_specs = grid_family_runs[grid_family_runs.family.eq("Beta")][
    ["dataset", "noise_type", "rho", "beta", "graph_neighbors", "gram_variant"]].drop_duplicates()
weight_sources = grid_runs.merge(weight_specs, on=["dataset", "noise_type", "rho", "graph_neighbors", "gram_variant"],
                                validate="many_to_one")
weight_columns = ["omega_margin", "omega_gradient", "omega_identity"]
weight_frames = []
for run in weight_sources.itertuples(index=False):
    weight_path = PROJECT_ROOT / Path(run.source).parent / "kernel_weights.csv"
    if not weight_path.is_file():
        raise FileNotFoundError(f"Missing weight trajectory: {weight_path}")
    w = pd.read_csv(weight_path)
    x = w.configured_budget_fraction.to_numpy(float)
    if (not np.allclose(x, np.r_[0., EXPECTED_BUDGETS[run.rho]])
        or not np.isfinite(w[weight_columns]).all().all()
        or not w[weight_columns].ge(0).all().all()
        or not np.allclose(w[weight_columns].sum(axis=1), 1, atol=1e-6)):
        raise ValueError(f"Invalid weight trajectory: {weight_path}")
    for key in ["dataset", "noise_type", "rho", "seed", "beta", "graph_neighbors"]:
        w[key] = getattr(run, key)
    weight_frames.append(w)
weights = pd.concat(weight_frames, ignore_index=True)
weight_keys = ["dataset", "rho", "noise_type", "beta", "graph_neighbors", "configured_budget_fraction"]
weight_seed_means = weights.groupby(weight_keys, as_index=False, observed=True)[weight_columns].mean()
weight_per_noise = weight_seed_means.groupby(
    ["dataset", "rho", "noise_type", "configured_budget_fraction"], as_index=False, observed=True)[weight_columns].mean()
weight_summary = weight_per_noise.groupby(
    ["dataset", "rho", "configured_budget_fraction"], observed=True)[weight_columns].agg(["mean", "std", "count"])
assert (weight_summary.xs("count", axis=1, level=1) == len(NOISE_TYPES)).all().all()
assert np.allclose(weight_summary.xs("mean", axis=1, level=1).sum(axis=1), 1)
record_table(weight_sources, "adaptive_weight_sources")
record_table(weight_per_noise, "adaptive_weights_marginal_per_noise")
record_table(weight_summary.reset_index(), "adaptive_weights_marginal_summary")

def plot_marginal_weights(rho):
    fig, axes = plt.subplots(2, 3, figsize=(8.6, 5.1), sharex=True, sharey=True, layout="constrained")
    for index, (ax, dataset) in enumerate(zip(axes.flat, DATASETS)):
        panel = weight_summary.loc[(dataset, rho)]
        x = panel.index.to_numpy(float)
        for component in ["margin", "gradient", "identity"]:
            mean = panel[(f"omega_{component}", "mean")].to_numpy(float)
            std = panel[(f"omega_{component}", "std")].to_numpy(float)
            color, marker = COMPONENT_COLORS[component], COMPONENT_MARKERS[component]
            ax.plot(x, mean, color=color, marker=marker, markersize=3.8, linewidth=2., label=component.capitalize())
            ax.fill_between(x, np.clip(mean-std, 0, 1), np.clip(mean+std, 0, 1), color=color, alpha=.11)
        ax.axhline(1/3, color="#777777", linestyle="--", linewidth=1.4, label="Uniform 1/3")
        ax.set_xlim(0, rho)
        ax.set_xticks(np.linspace(0, rho, 5))
        ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
        ax.set_ylim(0, 1)
        ax.set_title(DATASET_LABELS[dataset])
        ax.set_xlabel("Verification budget")
        if index % 3 == 0:
            ax.set_ylabel("Kernel weight")
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, frameon=False)
    fig.suptitle(f"Adaptive kernel weights | rho={rho:g}")
    return finish_marginal_figure(fig, f"marginal_adaptive_kernel_weights_rho{rho:g}",
        weight_per_noise[np.isclose(weight_per_noise.rho, rho)],
        "Average available seeds, then beta and k, then noise type; shaded bands show noise-type SD clipped to [0, 1].")

for rho in RHOS:
    plot_marginal_weights(rho)

In [ ]:
display(pd.DataFrame(FIGURE_MANIFEST))
assert len(GRID_PDF_PATHS) == 12
assert len(set(GRID_PDF_PATHS)) == len(GRID_PDF_PATHS)